# Semaine 2 — Jour 5 : Memory courte, longue et state

Ce notebook est la version étudiant du jour 5.

Il accompagne les fichiers Markdown de référence dans `book/week02/day05/`.

Objectif : comprendre et tester une architecture mémoire minimale pour un agent IA.

## 1. Modèle mental

Un agent mémoire-aware sépare trois responsabilités :

| Couche | Rôle |
|---|---|
| Short-term memory | Conserver les derniers échanges utiles |
| Conversation state | Suivre la tâche en cours |
| Long-term memory | Conserver des préférences ou faits stables par utilisateur |

La mémoire longue ne doit pas contenir tout l'historique.

## 2. Charger le lab

La cellule suivante cherche automatiquement le dossier du lab dans le dépôt.

In [ ]:
from pathlib import Path
import sys

def find_lab_path() -> Path:
    for candidate_root in [Path.cwd(), *Path.cwd().parents]:
        lab_path = candidate_root / "book" / "week02" / "day05" / "labs"
        if (lab_path / "memory_agent.py").exists():
            return lab_path
    raise FileNotFoundError("Impossible de trouver book/week02/day05/labs")

LAB_PATH = find_lab_path()
sys.path.insert(0, str(LAB_PATH))

from memory_agent import MemoryAwareSupportAgent, LongTermMemoryStore

LAB_PATH

## 3. Première interaction

On initialise un agent déterministe.

Aucun appel réseau n'est effectué.

In [ ]:
agent = MemoryAwareSupportAgent(short_term_max_messages=4)

print(agent.receive("user_1", "Tu peux m'appeler Nadia."))
print(agent.receive("user_1", "Je préfère les exemples en Python et les réponses courtes."))
print(agent.receive("user_1", "J'ai un problème avec Billing API."))

## 4. Observer le contexte construit

Le contexte contient :

- le profil utilisateur ;
- l'état courant ;
- les messages récents ;
- un résumé court.

In [ ]:
import json

context = agent.build_context("user_1")
print(json.dumps(context, ensure_ascii=False, indent=2))

## 5. Isolation utilisateur

Une mémoire professionnelle ne doit jamais mélanger les préférences de deux utilisateurs.

In [ ]:
isolated_agent = MemoryAwareSupportAgent()

isolated_agent.receive("alice", "Je préfère les exemples en Python.")
isolated_agent.receive("bob", "Je préfère les exemples en TypeScript.")

alice_response = isolated_agent.receive("alice", "Aide-moi à créer un ticket.")
bob_response = isolated_agent.receive("bob", "Aide-moi à créer un ticket.")

print("Alice:", alice_response)
print("Bob:", bob_response)

## 6. Oubli utilisateur

La mémoire doit pouvoir être supprimée.

`forget_user` supprime :

- mémoire longue ;
- mémoire courte ;
- state courant.

In [ ]:
isolated_agent.forget_user("alice")
print(json.dumps(isolated_agent.build_context("alice"), ensure_ascii=False, indent=2))

## 7. Lancer les tests

Les tests valident la couche applicative sans LLM.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "test_memory_agent.py"],
    cwd=str(LAB_PATH),
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0

## 8. Exercices

Travaille ensuite dans `exercises.md`.

Points prioritaires :

1. classer les informations dans la bonne couche mémoire ;
2. concevoir un profil JSON ;
3. définir une politique de promotion ;
4. écrire un test d'isolation utilisateur ;
5. proposer une architecture mémoire pour un assistant de documentation.

# Section formateur

Cette section contient les corrigés, indications de review et points d'évaluation.

# Corrigé — Exercices

## Exercice 1 — Classer les informations

| Élément | Classification | Justification |
|---|---|---|
| “L’utilisateur vient de demander un remboursement.” | conversation state | C’est l’intention active de la tâche. |
| `order_id` manque | conversation state | C’est un slot manquant. |
| Préférence pour Python | long-term memory | Préférence stable explicitement donnée. |
| Dernier outil en erreur 404 | short-term memory | Utile pour les prochains tours, pas durable. |
| Numéro de carte bancaire | ne pas mémoriser | Donnée sensible à refuser. |
| Appeler l’utilisateur Sam | long-term memory | Nom d’affichage explicitement donné. |
| Statut `waiting_for_user` | conversation state | État courant de la tâche. |
| Frustration dans le message précédent | short-term memory ou ne pas mémoriser | Utile à court terme pour le ton, pas durable. |

## Exercice 2 — Profil mémoire

Exemple :

```json
{
  "user_id": "user_123",
  "display_name": "Sam",
  "preferences": {
    "language": "Python",
    "answer_style": "concise"
  },
  "facts": [
    {
      "key": "main_framework",
      "value": "FastAPI",
      "source": "explicit_user_statement",
      "updated_at": "2026-08-24T14:00:00"
    }
  ],
  "updated_at": "2026-08-24T14:00:00"
}
```

L’historique complet n’est pas stocké dans le profil.

## Exercice 3 — Politique de promotion

| Phrase | Décision | Couche cible | Justification |
|---|---|---|---|
| Je préfère les réponses courtes. | stocker | long-term memory | Préférence explicite et utile. |
| Aujourd’hui je suis très fatigué. | ne pas stocker durablement | short-term memory éventuelle | Information temporaire. |
| Je travaille principalement avec FastAPI. | stocker | long-term memory | Fait stable utile pour personnaliser les exemples. |
| Mon mot de passe est hunter2. | refuser | aucune | Secret. |
| Pour ce ticket, le produit concerné est Billing API. | stocker pour la tâche | conversation state | Donnée liée à la tâche en cours. |
| Tu peux m’appeler Nadia. | stocker | long-term memory | Nom préféré explicite. |

## Exercice 4 — Lecture du code

1. La mémoire courte est limitée dans `ShortTermMemory.add`.
2. Le state est mis à jour dans `_update_state`.
3. L’isolation utilisateur est assurée par `LongTermMemoryStore._profiles[user_id]`.
4. L’oubli est implémenté dans `forget_user`.
5. Les tests n’appellent pas de LLM car ils valident la couche applicative déterministe.

## Exercice 5 — Test d’isolation

Exemple :

```python
def test_user_preferences_are_isolated():
    agent = MemoryAwareSupportAgent()

    agent.receive("alice", "Je préfère les exemples en Python")
    agent.receive("bob", "Je préfère les exemples en TypeScript")

    alice_response = agent.receive("alice", "Aide-moi à créer un ticket")
    bob_response = agent.receive("bob", "Aide-moi à créer un ticket")

    assert "Python" in alice_response
    assert "TypeScript" in bob_response
    assert "TypeScript" not in alice_response
    assert "Python" not in bob_response
```

## Exercice 6 — Contexte minimal

```text
Profil utilisateur:
- Nom préféré: Nadia
- Langage préféré: Python
- Style de réponse: concise

État courant:
- Intention: create_support_ticket
- Produit: Billing API
- Champ manquant: description
- Statut: collecting

Historique récent:
- user: L'API répond 500 depuis ce matin.
- assistant: Quel produit est concerné ?
- user: Billing API.

Instruction:
Demander la description du problème sans redemander le produit.
```

## Exercice 7 — Anti-patterns

1. Stocker tous les messages crée du bruit, du coût et des risques de données sensibles.
2. Une mémoire globale provoque des fuites entre utilisateurs.
3. Le modèle peut promouvoir des informations temporaires ou inférées à tort.
4. Sans suppression, la mémoire n’est pas gouvernable.
5. Réinjecter toute la mémoire augmente le coût et peut polluer la réponse.

## Exercice 8 — Mini-design

Architecture proposée :

```text
User request
→ ShortTermMemory(max_messages=8)
→ StateTracker(intent, slots, missing_slots)
→ MemoryExtractor(structured output)
→ MemoryPolicy
→ UserProfileStore(user_id)
→ ContextBuilder
→ Model call
```

Mémoire longue autorisée :

- stack préférée ;
- niveau d’expertise ;
- format de réponse.

Mémoire refusée :

- secrets ;
- informations sensibles ;
- erreurs temporaires ;
- transcript complet.

# Corrigé — Interview

## Réponse 1

La short-term memory conserve le contexte récent. Le conversation state décrit la tâche en cours. La long-term memory conserve des informations stables et réutilisables dans de futures conversations.

## Réponse 2

Il ne faut pas stocker tout l’historique car cela augmente le coût, ajoute du bruit, conserve potentiellement des données sensibles et rend la mémoire difficile à gouverner.

## Réponse 3

On isole la mémoire par identifiant utilisateur ou tenant. Les clés de stockage doivent inclure `user_id`, et les tests doivent vérifier qu’une préférence utilisateur n’est jamais visible pour un autre.

## Réponse 4

Une information promue vers la mémoire longue doit être explicite, stable, utile, non sensible et révocable.

## Réponse 5

Le conversation state est limité à une tâche. Il ne conserve pas nécessairement les préférences réutilisables entre sessions.

## Réponse 6

On teste la couche mémoire avec du code déterministe : ajout de messages, extraction simulée, mise à jour de profil, construction de contexte et suppression.

## Réponse 7

`forget_user(user_id)` doit supprimer le profil, l’historique court associé, les états actifs liés à l’utilisateur et les journaux d’audit éventuels.

## Réponse 8

Un résumé de conversation compresse des échanges récents ou passés. Une mémoire longue représente des faits ou préférences sélectionnés selon une politique.

## Réponse 9

Une politique explicite évite que des informations temporaires, sensibles ou inférées deviennent des vérités durables.

## Réponse 10

Il faut traiter la confidentialité, la rétention, la suppression, l’audit, le consentement, la sécurité, le contrôle utilisateur et la conformité.

## Réponse 11

Structured Outputs peut forcer l’extraction mémoire à produire un objet validable comme `{should_store, key, value, reason, confidence}`.

## Réponse 12

Une mémoire trop agressive peut personnaliser à tort, conserver des données sensibles, amplifier des erreurs et diminuer la confiance utilisateur.

# Corrigé — Challenge

## Approche attendue

Le challenge ajoute une couche de gouvernance entre l’extraction mémoire et l’écriture en mémoire longue.

Architecture :

```text
message utilisateur
→ extraction candidate
→ MemoryPolicy.evaluate(candidate)
→ audit
→ écriture conditionnelle
```

## Exemple de modèle de décision

```python
@dataclass
class MemoryDecision:
    allowed: bool
    reason: str
    memory_type: str | None = None
```

## Exemple de politique

```python
class MemoryPolicy:
    FORBIDDEN_PATTERNS = [
        "mot de passe",
        "password",
        "token",
        "api key",
        "carte bancaire",
        "malade",
        "déprimé",
    ]

    STATE_ONLY_PATTERNS = [
        "pour ce ticket",
        "commande",
        "order_id",
        "produit concerné",
    ]

    def evaluate(self, text: str, key: str | None = None) -> MemoryDecision:
        normalized = text.lower()

        if any(pattern in normalized for pattern in self.FORBIDDEN_PATTERNS):
            return MemoryDecision(False, "forbidden_sensitive_content")

        if any(pattern in normalized for pattern in self.STATE_ONLY_PATTERNS):
            return MemoryDecision(False, "state_only_information")

        if "je préfère" in normalized or "tu peux m'appeler" in normalized:
            return MemoryDecision(True, "explicit_preference", "preference")

        return MemoryDecision(False, "not_stable_or_not_useful")
```

## Audit attendu

Chaque tentative de mémorisation doit laisser une trace.

Exemple :

```json
{
  "user_id": "user_123",
  "source_text": "Je préfère les exemples en Python",
  "decision": "allowed",
  "reason": "explicit_preference",
  "memory_key": "language",
  "timestamp": "2026-08-24T14:00:00"
}
```

## Tests attendus

### 1. Préférence stockée

```python
def test_explicit_preference_is_stored():
    agent = GovernedMemoryAgent()
    agent.receive("u1", "Je préfère les exemples en Python")
    assert agent.memory.get_profile("u1").preferences["language"] == "Python"
```

### 2. Secret refusé

```python
def test_secret_is_rejected():
    agent = GovernedMemoryAgent()
    agent.receive("u1", "Mon mot de passe est hunter2")
    assert "hunter2" not in str(agent.memory.get_profile("u1").to_dict())
```

### 3. State non promu

```python
def test_ticket_data_is_not_long_term_memory():
    agent = GovernedMemoryAgent()
    agent.receive("u1", "Pour ce ticket, le produit concerné est Billing API")
    assert "Billing API" not in agent.memory.get_profile("u1").facts
```

### 4. Audit présent

```python
def test_audit_contains_decision():
    agent = GovernedMemoryAgent()
    agent.receive("u1", "Je préfère les réponses courtes")
    assert agent.audit_log[-1]["reason"] == "explicit_preference"
```

### 5. Oubli complet

```python
def test_forget_user_removes_audit():
    agent = GovernedMemoryAgent()
    agent.receive("u1", "Je préfère Python")
    agent.forget_user("u1")
    assert agent.audit_for("u1") == []
```

## Points de vigilance

- Ne pas ajouter de dépendance externe.
- Ne pas masquer les décisions de refus.
- Ne pas stocker le texte source complet si ce texte contient un secret.
- Préférer des champs structurés à une liste libre.
- Ajouter l’expiration uniquement lorsque cela clarifie le cycle de vie.

## Propositions d’amélioration

Ces propositions ne modifient pas les spécifications du bootcamp.

- Ajouter une UI de consultation mémoire utilisateur.
- Ajouter une stratégie d’expiration automatique.
- Ajouter une validation JSON Schema pour les décisions mémoire.
- Ajouter une métrique de précision d’extraction mémoire.

# Review formateur — Semaine 2 Jour 5

## Objectif de la review

Valider que l’apprenant comprend la mémoire comme une couche d’architecture et non comme une simple accumulation de messages.

## Points à vérifier

### Compréhension conceptuelle

L’apprenant doit distinguer :

- short-term memory ;
- conversation state ;
- long-term memory ;
- mémoire applicative ;
- historique brut.

### Compréhension technique

L’apprenant doit savoir expliquer :

- pourquoi `ShortTermMemory` limite le nombre de messages ;
- pourquoi `LongTermMemoryStore` est indexé par `user_id` ;
- pourquoi `ConversationState` ne stocke pas les préférences ;
- pourquoi `forget_user` est nécessaire ;
- pourquoi les tests sont déterministes.

### Qualité du code

Vérifier que :

- le code s’exécute sans dépendance externe ;
- les dataclasses sont lisibles ;
- les fonctions ont une responsabilité claire ;
- les tests couvrent l’isolation utilisateur ;
- la sérialisation ne casse pas les données.

## Questions de review

1. Quelle information du lab appartient au state et non à la mémoire longue ?
2. Pourquoi le nom préféré est-il une mémoire longue acceptable ?
3. Que faudrait-il changer avant d’utiliser ce système en production ?
4. Comment ajouter une politique de consentement utilisateur ?
5. Quelle différence entre `forget_user` et `clear_current_state` ?

## Attendus de réponse

L’apprenant doit répondre que :

- les données de ticket appartiennent au state ;
- les préférences explicites peuvent être stockées ;
- les secrets doivent être refusés ;
- l’isolation par utilisateur est obligatoire ;
- l’oubli est une fonctionnalité de gouvernance ;
- la mémoire longue doit rester sélective.

## Checklist de complétion

- [x] README présent
- [x] learning_objectives présent
- [x] chapter présent
- [x] exercises présent
- [x] interview présent
- [x] challenge présent
- [x] references présent
- [x] corrigés présents
- [x] diagrams présents
- [x] assets présents
- [x] labs présents
- [x] notebook étudiant généré
- [x] notebook formateur généré
- [x] code exécutable
- [x] tests automatisés

## Propositions d’amélioration

Ces propositions ne modifient pas les spécifications.

- Ajouter un exemple avec stockage SQLite en Semaine 4.
- Ajouter une démonstration de résumé incrémental en Semaine 8.
- Ajouter un mini-exercice sur consentement et suppression utilisateur.